# NOVA Remote GPU Worker — Google Colab

Один ноутбук для удалённого бесплатного рендера из NOVA: **Blender + FFmpeg + WanGP Python API**.

**iPhone → NOVA → HTTPS tunnel → Colab GPU → render → MP4 обратно на телефон.**

Запусти **Runtime → Run all**. В конце появятся `NOVA WORKER URL` и `NOVA TOKEN` — вставь их в Motion + VFX → Remote GPU.

> Colab Free не гарантирует GPU и может завершать сессии. Платные API этим ноутбуком не используются.

In [ ]:
# 1) Профиль NOVA — можно оставить как есть
import os, secrets, subprocess, sys, time, re, json, shutil
from pathlib import Path

PORT = 7861
ENABLE_WANGP = True       # Полностью автоматический headless WanGP API
USE_DRIVE_CACHE = False   # True: хранить WanGP weights/cache в Google Drive между сессиями
MIRROR_RESULTS_TO_DRIVE = False  # True: копировать completed jobs в MyDrive/NOVA_RENDER_QUEUE
TOKEN = secrets.token_urlsafe(24)

probe=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
print(probe.stdout if probe.stdout else 'GPU пока не обнаружен. Runtime → Change runtime type → GPU.')
if probe.returncode != 0:
    raise RuntimeError('GPU не найден. Выбери GPU runtime и запусти Run all снова.')
print('NOVA profile: preview-first, zero paid API, WanGP headless =', ENABLE_WANGP)


In [ ]:
# 2) NOVA worker stack + Blender + FFmpeg + pose tracking
env=os.environ.copy(); env['DEBIAN_FRONTEND']='noninteractive'
subprocess.run(['sudo','apt-get','update','-qq'],check=True,env=env)
subprocess.run(['sudo','apt-get','install','-y','--no-install-recommends','blender','ffmpeg','python3-venv'],check=True,env=env)
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn','python-multipart','mediapipe','opencv-python-headless','requests'],check=True)

REPO=Path('/content/nova-robot')
if (REPO/'.git').exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/magomedt149/nova-robot.git',str(REPO)],check=True)

subprocess.run([sys.executable,'-m','py_compile',str(REPO/'automation/remote_gpu_worker.py'),str(REPO/'automation/nova_pipeline.py'),str(REPO/'blender-colab/scripts/extract_pose.py')],check=True)
subprocess.run(['blender','--version'],check=True)
print('NOVA worker + Blender + FFmpeg: READY')


In [ ]:
# 3) Опциональный Google Drive cache/results
if USE_DRIVE_CACHE or MIRROR_RESULTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive mounted.')
else:
    print('Drive не нужен: файлы возвращаются напрямую в NOVA.')


In [ ]:
# 4) Полная установка WanGP API (без Gradio UI)
launch_code=None
if ENABLE_WANGP:
    UPSTREAM_DIR=Path('/content/Wan2GP-on-Colab')
    UPSTREAM_COMMIT='e428b5ebc0d49589474ef5d81e05cc2ab3c1e17b'
    if not (UPSTREAM_DIR/'.git').exists():
        subprocess.run(['git','clone','https://github.com/Square-Zero-Labs/Wan2GP-on-Colab.git',str(UPSTREAM_DIR)],check=True)
    subprocess.run(['git','-C',str(UPSTREAM_DIR),'fetch','--depth','1','origin',UPSTREAM_COMMIT],check=True)
    subprocess.run(['git','-C',str(UPSTREAM_DIR),'checkout','--detach',UPSTREAM_COMMIT],check=True)
    upstream_notebook=UPSTREAM_DIR/'wan2gp-google-colab.ipynb'
    upstream=json.loads(upstream_notebook.read_text(encoding='utf-8'))
    for number, cell in enumerate(upstream['cells']):
        if cell.get('cell_type')!='code':
            continue
        code=''.join(cell.get('source') or [])
        if 'Launching Wan2GP' in code:
            launch_code=code
            continue
        code=code.replace('USE_GOOGLE_DRIVE_DATA = False', f'USE_GOOGLE_DRIVE_DATA = {USE_DRIVE_CACHE!r}')
        print(f'\n--- WanGP setup {number} ---')
        exec(compile(code, f'{upstream_notebook.name}:cell_{number}', 'exec'), globals())
    WAN2GP_ROOT=Path('/content/Wan2GP')
    api_file=WAN2GP_ROOT/'shared/api.py'
    if not api_file.is_file():
        raise RuntimeError('WanGP shared/api.py not found after setup.')
    os.environ['NOVA_WANGP_ROOT']=str(WAN2GP_ROOT)
    print('WanGP Python API: READY', api_file)
else:
    print('WanGP disabled. Blender/FFmpeg still work.')


In [ ]:
# 5) START NOVA GPU WORKER + защищённый HTTPS tunnel
import urllib.request
cloudflared=Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)

worker_log=Path('/content/nova_worker.log'); tunnel_log=Path('/content/nova_tunnel.log')
os.environ['NOVA_REMOTE_TOKEN']=TOKEN
os.environ['NOVA_REMOTE_JOB_ROOT']='/content/NOVA_REMOTE_JOBS'
if ENABLE_WANGP:
    os.environ['NOVA_WANGP_ROOT']='/content/Wan2GP'
if MIRROR_RESULTS_TO_DRIVE:
    os.environ['NOVA_REMOTE_DRIVE_ROOT']='/content/drive/MyDrive/NOVA_RENDER_QUEUE'

# остановить старые процессы этой сессии, если клетка запускается повторно
for name in ('worker','tunnel'):
    old=globals().get(name)
    if old is not None and getattr(old,'poll',lambda:0)() is None:
        try: old.terminate()
        except Exception: pass

worker=subprocess.Popen([sys.executable,str(REPO/'automation/remote_gpu_worker.py'),'--host','0.0.0.0','--port',str(PORT)],stdout=worker_log.open('w'),stderr=subprocess.STDOUT,env=os.environ.copy())
time.sleep(4)
if worker.poll() is not None:
    print(worker_log.read_text(errors='replace'))
    raise RuntimeError('NOVA worker did not start')

tunnel=subprocess.Popen([str(cloudflared),'tunnel','--url',f'http://127.0.0.1:{PORT}','--no-autoupdate'],stdout=tunnel_log.open('w'),stderr=subprocess.STDOUT,text=True)
url=None
for _ in range(60):
    time.sleep(1)
    txt=tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',txt)
    if m:
        url=m.group(0); break
if not url:
    print(tunnel_log.read_text(errors='replace'))
    raise RuntimeError('Cloudflare tunnel URL not found')

import requests
health=requests.get(url+'/health',headers={'X-NOVA-Token':TOKEN},timeout=30).json()
print('\n'+'='*78)
print('NOVA WORKER URL:',url)
print('NOVA TOKEN     :',TOKEN)
connect_code='NOVA_CONNECT='+json.dumps({'url':url,'token':TOKEN},separators=(',',':'))
print('NOVA CONNECT CODE:',connect_code)
print('='*78)
print(json.dumps(health,ensure_ascii=False,indent=2))
print('\nГОТОВО: Motion + VFX → Remote GPU → вставь URL и Token → Проверить GPU.')


## Готово

Пока последняя клетка и Colab runtime остаются запущенными, NOVA может отправлять задания с телефона. **Blender, FFmpeg и WanGP теперь запускаются без ручной Gradio-генерации.**

На T4 NOVA выбирает облегчённую WanGP-модель (приоритет FastWan), генерирует экономно и для `Final` перекодирует результат в H.264 1080p. На более мощном GPU модель выбирается с учётом доступной VRAM и video-input возможностей.